# 🤖 RAG-Powered AI Study Assistant — walkthrough

**GenAI Stack:** LlamaIndex · FAISS · Ollama (Mistral / Qwen) · Streamlit

This notebook walks through every stage of the Retrieval-Augmented Generation
(RAG) pipeline used by `RAG_app.py`, using a fully open-source stack that runs
locally with no paid APIs.

## Table of Contents

1. [Why RAG?](#why)
2. [The GenAI stack & data flow](#stack)
3. [Setup](#setup)
4. [Step 1 — Load documents](#load)
5. [Step 2 — Chunk into nodes](#chunk)
6. [Step 3 — Embeddings](#embed)
7. [Step 4 — Build the FAISS vector store](#faiss)
8. [Step 5 — Retrieval](#retrieve)
9. [Step 6 — Generation with Ollama (Mistral / Qwen)](#generate)
10. [Step 7 — Conversational engine with memory & citations](#chat)
11. [Persisting & reloading the index](#persist)
12. [The Streamlit app](#app)
13. [Conclusion](#conclusion)

# <a id="why">Why RAG?</a>

Large Language Models are powerful but they **hallucinate** and go **stale** —
they only know their training data. Ask an open-source model about *your* course
materials and it will confidently invent an answer.

**Retrieval-Augmented Generation** fixes this by grounding the model in your own
documents: we chunk and embed the documents, retrieve the most relevant passages
for each question, and let the LLM answer using *only* that retrieved context —
citing its sources so the answer stays faithful to the material.

# <a id="stack">The GenAI stack & data flow</a>

```
        ┌───────────────┐  load   ┌──────────────┐ chunk  ┌──────────────┐
Upload  │ SimpleDirectory│ ──────► │ SentenceSplit│ ─────► │  HuggingFace │
docs ──►│    Reader      │         │ (chunk+over) │        │  embeddings  │
        └───────────────┘         └──────────────┘        └──────┬───────┘
                                                                 │ embed
                                                                 ▼
   ┌─────────────────────────────────────────────┐       ┌──────────────┐
   │ Ollama LLM (Mistral / Qwen)                 │context│     FAISS    │
   │ "answer using ONLY this context, cite it"   │◄──────│ vector index │
   └──────────────────────┬──────────────────────┘ top-k │  (on disk)   │
                          │ source-cited answer    retrieve└─────▲───────┘
                          ▼                                      │ embed query
                ┌──────────────────┐                            │
                │  Streamlit chat  │◄────────────  user question┘
                └──────────────────┘
```

| Stage | Component | Library |
|-------|-----------|---------|
| Load | `SimpleDirectoryReader` | LlamaIndex |
| Chunk | `SentenceSplitter` | LlamaIndex |
| Embed | `HuggingFaceEmbedding` | LlamaIndex + sentence-transformers |
| Store | `FaissVectorStore` | FAISS |
| Retrieve | top-k similarity search | FAISS |
| Generate | `Ollama` (Mistral / Qwen) | LlamaIndex + Ollama |
| Serve | chat UI | Streamlit |

# <a id="setup">Setup</a>

Install the dependencies and make sure [Ollama](https://ollama.com) is running
with an open-source model pulled.

In [ ]:
# !pip install -r requirements.txt
# Then, with Ollama installed (https://ollama.com):
#   ollama pull mistral
#   ollama pull qwen2.5

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import faiss

from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    SimpleDirectoryReader,
    load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.vector_stores.faiss import FaissVectorStore

In [ ]:
# Config
DATA_DIR = Path("data/docs")                 # source documents
PERSIST_DIR = "data/vector_stores/notebook"  # where the FAISS index is saved

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # local, 384-dim
LLM_MODEL = "mistral"                         # or "qwen2.5", served by Ollama

# <a id="load">Step 1 — Load documents</a>

`SimpleDirectoryReader` reads every supported file (pdf, txt, docx, csv, md …)
into LlamaIndex `Document` objects, attaching metadata such as the file name and
page label that we will later surface as **source citations**.

In [ ]:
documents = SimpleDirectoryReader(
    input_files=[str(DATA_DIR / "2024-01-24-VirtualTryAll.pdf")]
).load_data()

print(f"Loaded {len(documents)} document object(s)")
print("Metadata of first page:", documents[0].metadata)
print("\nPreview:\n", documents[0].text[:400])

# <a id="chunk">Step 2 — Chunk into nodes</a>

LLMs have limited context windows, and retrieval is more precise on small
passages. `SentenceSplitter` splits documents into overlapping **nodes** — the
overlap keeps sentences that straddle a boundary from losing their context.

In [ ]:
splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
nodes = splitter.get_nodes_from_documents(documents)

print(f"{len(documents)} document(s) -> {len(nodes)} chunk(s)")
print("\nExample chunk:\n", nodes[0].get_content()[:400])

# <a id="embed">Step 3 — Embeddings</a>

An embedding model maps text to a dense vector so that semantically similar text
ends up close together. We use a **local** HuggingFace model — no API key, runs
on CPU. The model is downloaded once and cached.

In [ ]:
embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

vec = embed_model.get_text_embedding("What is Diffuse to Choose?")
print(f"Embedding dimension: {len(vec)}")

In [ ]:
# Quick sanity check: similar sentences score higher (cosine similarity)
import numpy as np

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

e1 = embed_model.get_text_embedding("I like pets.")
e2 = embed_model.get_text_embedding("I love animals.")
e3 = embed_model.get_text_embedding("The stock market fell today.")

print("pets vs animals :", round(cos(e1, e2), 3))
print("pets vs stocks  :", round(cos(e1, e3), 3))

# <a id="faiss">Step 4 — Build the FAISS vector store</a>

[FAISS](https://github.com/facebookresearch/faiss) is a high-performance library
for similarity search over dense vectors. We create a flat L2 index sized to the
embedding dimension, wrap it in LlamaIndex's `FaissVectorStore`, and build the
index — this embeds every node and adds its vector to FAISS.

In [ ]:
dim = len(vec)                          # embedding dimension from Step 3
faiss_index = faiss.IndexFlatL2(dim)    # exact L2 nearest-neighbour search
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
)
print("FAISS index now holds", faiss_index.ntotal, "vectors")

# <a id="retrieve">Step 5 — Retrieval</a>

A retriever embeds the query and asks FAISS for the top-k nearest chunks. These
are the passages that will be handed to the LLM as context.

In [ ]:
retriever = index.as_retriever(similarity_top_k=4)
results = retriever.retrieve("What is Diffuse to Choose?")

for i, node in enumerate(results, 1):
    src = node.node.metadata.get("file_name", "?")
    print(f"[{i}] score={node.score:.3f}  source={src}")
    print("    ", node.node.get_content()[:160].replace("\n", " "), "...\n")

# <a id="generate">Step 6 — Generation with Ollama (Mistral / Qwen)</a>

Now we connect the open-source LLM. Ollama serves the model locally; LlamaIndex's
query engine retrieves context and asks the model to answer from it. Setting a
low temperature keeps answers grounded.

In [ ]:
llm = Ollama(model=LLM_MODEL, request_timeout=180.0, temperature=0.1)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=4)
response = query_engine.query("What is Diffuse to Choose? Answer concisely.")

print(response.response)
print("\n--- cited from ---")
for node in response.source_nodes:
    print(" -", node.node.metadata.get("file_name", "?"),
          "(score", round(node.score, 3), ")")

# <a id="chat">Step 7 — Conversational engine with memory & citations</a>

For a chatbot we need **memory** so follow-up questions resolve pronouns like
"it". The `condense_plus_context` engine rewrites the follow-up into a standalone
question, retrieves fresh context, and answers — while a context prompt forces
the model to answer only from the sources (the anti-hallucination guardrail).

In [ ]:
CONTEXT_PROMPT = (
    "You are a study assistant for technical course materials. Use ONLY the "
    "context below to answer. If the answer is not in the context, say you "
    "don't know.\n----------------------\n{context_str}\n"
    "----------------------\n"
)

memory = ChatMemoryBuffer.from_defaults(token_limit=3000)
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    llm=llm,
    memory=memory,
    context_prompt=CONTEXT_PROMPT,
    similarity_top_k=4,
)

In [ ]:
r1 = chat_engine.chat("What is Diffuse to Choose?")
print("Q1:", r1.response, "\n")

# Follow-up — "it" is resolved using chat memory
r2 = chat_engine.chat("What are its main use cases?")
print("Q2:", r2.response)

# <a id="persist">Persisting & reloading the index</a>

The FAISS index and docstore are saved to disk so the app can reload them
instantly instead of re-embedding every document.

In [ ]:
# Persist
index.storage_context.persist(persist_dir=PERSIST_DIR)
print("Saved to", PERSIST_DIR)

# Reload
vs = FaissVectorStore.from_persist_dir(PERSIST_DIR)
sc = StorageContext.from_defaults(vector_store=vs, persist_dir=PERSIST_DIR)
reloaded = load_index_from_storage(sc, embed_model=embed_model)
print("Reloaded index OK")

# <a id="app">The Streamlit app</a>

`RAG_app.py` wraps this exact pipeline in a Streamlit UI: upload documents, build
or load a FAISS vectorstore, then chat with your data. Every answer ships with an
expandable **Source documents** panel showing the retrieved chunks and scores.

Run it with:

```bash
streamlit run RAG_app.py
```

# <a id="conclusion">Conclusion</a>

We built a full RAG pipeline on an open-source GenAI stack — **LlamaIndex** for
ingestion, **FAISS** for vector similarity search, **Ollama (Mistral / Qwen)**
for source-cited generation, and **Streamlit** for the UI — with no paid APIs.
Because answers are grounded in retrieved context and shown with their sources,
the assistant stays faithful to the course material instead of hallucinating.